# Graph2Vec: Whole-Graph Embeddings

## Learning Objectives

1. **Contrast** node embeddings vs whole-graph embeddings
2. **Explain** the Weisfeiler-Lehman (WL) subgraph kernel
3. **Describe** Graph2Vec's Doc2Vec-inspired algorithm
4. **Apply** whole-graph embeddings to graph classification

## Why Whole-Graph Embeddings?

Node embeddings represent individual nodes. For tasks like:
- **Graph classification:** is this molecule toxic?
- **Graph similarity:** how similar are two program call graphs?
- **Graph retrieval:** find graphs structurally similar to a query

We need a **single vector** representing the entire graph.

**Approach 1:** Pool node embeddings (mean/max/sum) — simple but loses structural information.

**Approach 2:** Compute a graph-level embedding directly via Graph2Vec.

## Weisfeiler-Lehman (WL) Rooted Subgraphs

The WL graph isomorphism test provides a way to capture graph structure.

**WL labeling procedure:**
1. Initialize: each node $v$ gets label $l_0(v)$ = its degree (or original feature)
2. For $h = 1, 2, \ldots, H$:
   - New label $l_h(v) = \text{hash}(l_{h-1}(v), \text{sorted}(\{l_{h-1}(u): u \in N(v)\}))$
3. The **rooted subgraph** of depth $h$ around $v$ is represented by $l_h(v)$

**Graph2Vec** treats each WL-relabeled subgraph as a "word" and each graph as a "document".
It trains Doc2Vec (skip-gram) to embed graphs into $\mathbb{R}^d$.

In [ ]:
from collections import defaultdict
import hashlib

def wl_relabeling(adj, n_nodes, H=2, initial_labels=None):
    """WL relabeling: return list of (graph_id='', label_sets)."""
    if initial_labels is None:
        # Use degree as initial label
        labels = {v: len(adj[v]) for v in range(n_nodes)}
    else:
        labels = dict(initial_labels)

    all_labels = [set(labels.values())]  # h=0 labels
    for h in range(H):
        new_labels = {}
        for v in range(n_nodes):
            nbr_labels = sorted([labels[u] for u in adj[v]])
            # Hash current label + sorted neighbor labels
            key = f"{labels[v]},{nbr_labels}"
            new_labels[v] = int(hashlib.md5(key.encode()).hexdigest(), 16) % 10000
        labels = new_labels
        all_labels.append(set(labels.values()))
    return all_labels

# Three small graphs
def make_adj(edges, n):
    adj = defaultdict(set)
    for u,v in edges:
        adj[u].add(v); adj[v].add(u)
    return adj

graphs = [
    make_adj([(0,1),(1,2),(2,3),(3,0)], 4),    # Square
    make_adj([(0,1),(1,2),(2,0),(3,0)], 4),     # Triangle + tail
    make_adj([(0,1),(1,2),(2,3),(3,4),(4,0),(0,2)], 5),  # Pentagon + chord
]
n_nodes_list = [4, 4, 5]

print("WL label sets per graph (h=0,1,2):")
for i, (adj, n) in enumerate(zip(graphs, n_nodes_list)):
    label_sets = wl_relabeling(adj, n, H=2)
    print(f"  Graph {i}: h=0: {label_sets[0]}  h=1: {label_sets[1]}")

In [ ]:
# Simulate Graph2Vec: represent each graph as a bag of WL subgraph labels
# In practice, Doc2Vec learns embeddings; here we use feature hashing for illustration

def graph_wl_features(adj, n_nodes, H=2):
    """Flatten all WL labels across depths into a multiset."""
    label_sets = wl_relabeling(adj, n_nodes, H)
    features = []
    for h, lbls in enumerate(label_sets):
        for lbl in lbls:
            features.append(f"h{h}_{lbl}")
    return features

all_features = [graph_wl_features(adj, n, H=2) for adj, n in zip(graphs, n_nodes_list)]
vocab = sorted(set(f for feats in all_features for f in feats))
print(f"Vocabulary of WL subgraph patterns: {len(vocab)}")

# Represent each graph as a binary feature vector
def to_vec(features, vocab):
    vset = set(features)
    return [1 if f in vset else 0 for f in vocab]

vecs = [to_vec(feats, vocab) for feats in all_features]

def cosine(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = sum(x**2 for x in a)**0.5; nb = sum(x**2 for x in b)**0.5
    return dot/(na*nb) if na*nb > 0 else 0

print("
Graph similarity (WL feature cosine):")
names = ['Square','Triangle+tail','Pentagon+chord']
for i in range(3):
    for j in range(i+1, 3):
        sim = cosine(vecs[i], vecs[j])
        print(f"  {names[i]} vs {names[j]}: {sim:.4f}")

## Graph2Vec Algorithm Summary

1. For each graph $G_k$ in the corpus:
   - Extract all depth-$H$ WL rooted subgraphs
   - Treat the graph as a "document" and subgraphs as "words"

2. Train Doc2Vec (PV-DM or PV-DBOW) on these document-word pairs
   - Output: embedding $z_k \in \mathbb{R}^d$ for each graph

3. Use $z_k$ for downstream tasks:
   - **Classification:** SVM or logistic regression on $z_k$
   - **Clustering:** k-means on $z_k$
   - **Retrieval:** nearest neighbors in embedding space

**Advantage:** unsupervised; learns graph-level representations from structure alone.
**Limitation:** relies on node label information; graphs without meaningful node labels may produce poor embeddings.